# Day 09. Exercise 03
# Ensembles

## 0. Imports

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import joblib
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import ParameterGrid, cross_val_score
from tqdm.notebook import tqdm
import importlib
import tqdm.notebook as tqdm_notebook
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score
)
from sklearn.ensemble import VotingClassifier
from sklearn.ensemble import BaggingClassifier
import warnings

warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    message=r"The `probability` parameter was deprecated.*"
)
from sklearn.ensemble import StackingClassifier
from sklearn.multiclass import OneVsRestClassifier

## 1. Preprocessing

1. Create the same dataframe as in the previous exercise.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test` and then get `X_train`, `y_train`, `X_valid`, `y_valid` from the previous `X_train`, `y_train`. Use the additional parameter `stratify`.

In [38]:
df = pd.read_csv("../data/day-of-week-not-scaled.csv")
old_df = pd.read_csv("../data/dayofweek.csv")
df["dayofweek"] = old_df["dayofweek"]
X = df.drop(columns=["dayofweek"])
y = df["dayofweek"]

In [39]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=21,
    stratify=y
)

In [40]:
X_train_s, X_valid_s, y_train_s, y_valid_s = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=21,
    stratify=y_train
)

## 2. Individual classifiers

1. Train SVM, decision tree and random forest again with the best parameters that you got from the 01 exercise with `random_state=21` for all of them.
2. Evaluate `accuracy`, `precision`, and `recall` for them on the validation set.
3. The result of each cell of the section should look like this:

```
accuracy is 0.87778
precision is 0.88162
recall is 0.87778
```

In [41]:
svm_model = SVC(
    kernel="rbf",
    C=10,
    gamma="auto",
    class_weight=None,
    probability=True,
    random_state=21
)

tree_model = DecisionTreeClassifier(
    max_depth=20,
    criterion="entropy",
    class_weight="balanced",
    random_state=21
)

forest_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=32,
    criterion="entropy",
    class_weight=None,
    random_state=21
)

svm_model.fit(X_train_s, y_train_s)
tree_model.fit(X_train_s, y_train_s)
forest_model.fit(X_train_s, y_train_s)

svm_pred = svm_model.predict(X_valid_s)
tree_pred = tree_model.predict(X_valid_s)
forest_pred = forest_model.predict(X_valid_s)

svm_ac = accuracy_score(y_valid_s, svm_pred)
svm_prec = precision_score(y_valid_s, svm_pred, average="weighted")
svm_rec = recall_score(y_valid_s, svm_pred, average="weighted")

tree_ac = accuracy_score(y_valid_s, tree_pred)
tree_prec = precision_score(y_valid_s, tree_pred, average="weighted")
tree_rec = recall_score(y_valid_s, tree_pred, average="weighted")

forest_ac = accuracy_score(y_valid_s, forest_pred)
forest_prec = precision_score(y_valid_s, forest_pred, average="weighted")
forest_rec = recall_score(y_valid_s, forest_pred, average="weighted")

print(
    f"""
svm accuracy is {svm_ac:.5f}
svm precision is {svm_prec:.5f} 
svm recall is {svm_rec:.5f}

tree accuracy is {tree_ac:.5f}
tree precision is {tree_prec:.5f} 
tree recall is {tree_rec:.5f}

forest accuracy is {forest_ac:.5f}
forest precision is {forest_prec:.5f} 
forest recall is {forest_rec:.5f}
"""
)




svm accuracy is 0.87778
svm precision is 0.88162 
svm recall is 0.87778

tree accuracy is 0.85185
tree precision is 0.85481 
tree recall is 0.85185

forest accuracy is 0.88889
forest precision is 0.88956 
forest recall is 0.88889



## 3. Voting classifiers

1. Using `VotingClassifier` and the three models that you have just trained, calculate the `accuracy`, `precision`, and `recall` on the validation set.
2. Play with the other parameteres.
3. Calculate the `accuracy`, `precision` and `recall` on the test set for the model with the best weights in terms of accuracy (if there are several of them with equal values, choose the one with the higher precision).

In [42]:
voting_model = VotingClassifier(
    estimators=[
        ("svm", svm_model),
        ("tree", tree_model),
        ("forest", forest_model)
    ],
    voting="soft",
    weights=[1, 1, 1]
)

voting_model.fit(X_train_s, y_train_s)
voting_pred = voting_model.predict(X_valid_s)

ac = accuracy_score(y_valid_s, voting_pred)
prec = precision_score(y_valid_s, voting_pred, average="weighted")
rec = recall_score(y_valid_s, voting_pred, average="weighted")

print(
    f"""
accuracy is {ac:.5f}
precision is {prec:.5f}
recall is {rec:.5f}
"""
)



accuracy is 0.88148
precision is 0.88371
recall is 0.88148



In [43]:
weights_list = [
    [1, 1, 1],
    [2, 1, 1],
    [1, 2, 1],
    [1, 1, 2],
    [3, 1, 1],
    [1, 3, 1],
    [1, 1, 3],
    [2, 2, 1],
    [2, 1, 2],
    [1, 2, 2]
]

voting_results = []
best_voting_model = None
best_voting_score = (-1, -1)

for voting in ["hard", "soft"]:
    for weights in weights_list:
        model = VotingClassifier(
            estimators=[
                ("svm", svm_model),
                ("tree", tree_model),
                ("forest", forest_model)
            ],
            voting=voting,
            weights=weights
        )

        model.fit(X_train_s, y_train_s)
        y_pred = model.predict(X_valid_s)

        ac = accuracy_score(y_valid_s, y_pred)
        prec = precision_score(y_valid_s, y_pred, average="weighted")
        rec = recall_score(y_valid_s, y_pred, average="weighted")

        voting_results.append({
            "voting": voting,
            "weights": weights,
            "accuracy": ac,
            "precision": prec,
            "recall": rec
        })

        if (ac, prec) > best_voting_score:
            best_voting_score = (ac, prec)
            best_voting_model = model


In [44]:
voting_results = pd.DataFrame(voting_results)
voting_results = voting_results.sort_values(
    ["accuracy", "precision"], ascending=False, kind="stable"
)
print(voting_results.to_string(index=False, float_format="{:.5f}".format))


voting   weights  accuracy  precision  recall
  soft [3, 1, 1]   0.90370    0.90594 0.90370
  soft [2, 1, 2]   0.90370    0.90366 0.90370
  soft [2, 1, 1]   0.90000    0.90156 0.90000
  soft [1, 1, 2]   0.90000    0.90082 0.90000
  hard [1, 2, 2]   0.90000    0.89991 0.90000
  hard [2, 2, 1]   0.89630    0.89565 0.89630
  soft [1, 1, 3]   0.89259    0.89255 0.89259
  hard [1, 1, 1]   0.89259    0.89232 0.89259
  hard [2, 1, 2]   0.89259    0.89232 0.89259
  hard [2, 1, 1]   0.88889    0.89308 0.88889
  hard [1, 2, 1]   0.88889    0.88959 0.88889
  hard [1, 1, 3]   0.88889    0.88956 0.88889
  hard [1, 1, 2]   0.88889    0.88901 0.88889
  soft [1, 1, 1]   0.88148    0.88371 0.88148
  hard [3, 1, 1]   0.87778    0.88162 0.87778
  soft [2, 2, 1]   0.87037    0.87303 0.87037
  soft [1, 2, 2]   0.87037    0.87272 0.87037
  hard [1, 3, 1]   0.85185    0.85481 0.85185
  soft [1, 2, 1]   0.85185    0.85481 0.85185
  soft [1, 3, 1]   0.85185    0.85481 0.85185


In [45]:
best_voting_pred = best_voting_model.predict(X_valid_s)

best_voting_ac = accuracy_score(y_valid_s, best_voting_pred)
best_voting_prec = precision_score(y_valid_s, best_voting_pred, average="weighted")
best_voting_rec = recall_score(y_valid_s, best_voting_pred, average="weighted")

print(f"voting is {best_voting_model.voting}")
print(f"weights are {best_voting_model.weights}")
print(
    f"""
validation accuracy is {best_voting_ac:.5f}
validation precision is {best_voting_prec:.5f}
validation recall is {best_voting_rec:.5f}
"""
)


voting is soft
weights are [3, 1, 1]

validation accuracy is 0.90370
validation precision is 0.90594
validation recall is 0.90370



In [46]:
voting_test_pred = best_voting_model.predict(X_test)

voting_test_ac = accuracy_score(y_test, voting_test_pred)
voting_test_prec = precision_score(y_test, voting_test_pred, average="weighted")
voting_test_rec = recall_score(y_test, voting_test_pred, average="weighted")

print(
    f"""
test accuracy is {voting_test_ac:.5f}
test precision is {voting_test_prec:.5f}
test recall is {voting_test_rec:.5f}
"""
)



test accuracy is 0.90237
test precision is 0.90300
test recall is 0.90237



Лучший вариант: voting="soft", weights=[3, 1, 1]. Веса указаны в порядке SVM, дерево, лес. Accuracy на валидации — 0.90370, precision — 0.90594. Модель выбрана по accuracy на валидации, при равенстве — по precision. Accuracy на тесте — 0.90237.

## 4. Bagging classifiers

1. Using `BaggingClassifier` and `SVM` with the best parameters create an ensemble, try different values of the `n_estimators`, use `random_state=21`.
2. Play with the other parameters.
3. Calculate the `accuracy`, `precision`, and `recall` for the model with the best parameters (in terms of accuracy) on the test set (if there are several of them with equal values, choose the one with the higher precision)

In [47]:
bagging_model = BaggingClassifier(
    estimator=svm_model,
    n_estimators=10,
    random_state=21
)

In [48]:
bagging_model.fit(X_train_s, y_train_s)
bagging_pred = bagging_model.predict(X_valid_s)

bagging_ac = accuracy_score(y_valid_s, bagging_pred)
bagging_prec = precision_score(y_valid_s, bagging_pred, average="weighted")
bagging_rec = recall_score(y_valid_s, bagging_pred, average="weighted")

print(
    f"""
accuracy is {bagging_ac:.5f}
precision is {bagging_prec:.5f}
recall is {bagging_rec:.5f}
"""
)


accuracy is 0.88519
precision is 0.89427
recall is 0.88519



In [49]:
n_estimators = [5,10,20]
max_samples = [0.5,0.8,1.0]
max_features = [0.8,1.0]

for estimator in n_estimators:
    for sample in max_samples:
        for feature in max_features:
            model = BaggingClassifier(
                estimator=svm_model,
                n_estimators=estimator,
                max_samples=sample,
                max_features=feature,
                random_state=21
            )
            model.fit(X_train_s, y_train_s)
            pred = model.predict(X_valid_s)
            ac = accuracy_score(y_valid_s, pred)
            print(
                f"acc for sample {sample}, model {estimator} "
                f"and feature {feature} is {ac:.5f}"
            )


acc for sample 0.5, model 5 and feature 0.8 is 0.63333
acc for sample 0.5, model 5 and feature 1.0 is 0.77037
acc for sample 0.8, model 5 and feature 0.8 is 0.70000
acc for sample 0.8, model 5 and feature 1.0 is 0.85926
acc for sample 1.0, model 5 and feature 0.8 is 0.71481
acc for sample 1.0, model 5 and feature 1.0 is 0.87407
acc for sample 0.5, model 10 and feature 0.8 is 0.76296
acc for sample 0.5, model 10 and feature 1.0 is 0.82593
acc for sample 0.8, model 10 and feature 0.8 is 0.81852
acc for sample 0.8, model 10 and feature 1.0 is 0.86667
acc for sample 1.0, model 10 and feature 0.8 is 0.82593
acc for sample 1.0, model 10 and feature 1.0 is 0.88519
acc for sample 0.5, model 20 and feature 0.8 is 0.79630
acc for sample 0.5, model 20 and feature 1.0 is 0.81852
acc for sample 0.8, model 20 and feature 0.8 is 0.83704
acc for sample 0.8, model 20 and feature 1.0 is 0.87037
acc for sample 1.0, model 20 and feature 0.8 is 0.85556
acc for sample 1.0, model 20 and feature 1.0 is 0.8851

In [50]:
best_bagging_model = BaggingClassifier(
    estimator=svm_model,
    n_estimators=10,
    max_samples=1.0,
    max_features=1.0,
    random_state=21
)

best_bagging_model.fit(X_train_s, y_train_s)
bagging_test_pred = best_bagging_model.predict(X_test)

bagging_test_ac = accuracy_score(y_test, bagging_test_pred)
bagging_test_prec = precision_score(y_test, bagging_test_pred, average="weighted")
bagging_test_rec = recall_score(y_test, bagging_test_pred, average="weighted")

print(
    f"""
test accuracy is {bagging_test_ac:.5f}
test precision is {bagging_test_prec:.5f}
test recall is {bagging_test_rec:.5f}
"""
)



test accuracy is 0.86391
test precision is 0.86966
test recall is 0.86391



Лучший вариант: n_estimators=10, max_samples=1.0, max_features=1.0. У моделей с 10 и 20 SVM одинаковая accuracy на валидации — 0.88519. Выбрана модель с 10 SVM, так как её precision выше: 0.89427 против 0.89258.

## 5. Stacking classifiers

1. To achieve reproducibility in this case you will have to create an object of cross-validation generator: `StratifiedKFold(n_splits=n, shuffle=True, random_state=21)`, where `n` you will try to optimize (the details are below).
2. Using `StackingClassifier` and the three models that you have recently trained, calculate the `accuracy`, `precision` and `recall` on the validation set, try different values of `n_splits` `[2, 3, 4, 5, 6, 7]` in the cross-validation generator and parameter `passthrough` in the classifier itself,
3. Calculate the `accuracy`, `precision`, and `recall` for the model with the best parameters (in terms of accuracy) on the test set (if there are several of them with equal values, choose the one with the higher precision). Use `final_estimator=LogisticRegression(solver='liblinear')`.

In [52]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=21)


In [53]:
stacking_model = StackingClassifier(
    estimators=[
        ("svm", svm_model),
        ("tree", tree_model),
        ("forest", forest_model)
    ],
    final_estimator=OneVsRestClassifier(
        LogisticRegression(
            solver="liblinear",
            random_state=21
        )
    ),
    cv=cv,
    passthrough=False
)

In [54]:
stacking_model.fit(X_train_s, y_train_s)
stacking_pred = stacking_model.predict(X_valid_s)

stacking_ac = accuracy_score(y_valid_s, stacking_pred)
stacking_prec = precision_score(
    y_valid_s, stacking_pred, average="weighted"
)
stacking_rec = recall_score(
    y_valid_s, stacking_pred, average="weighted"
)

print(
    f"""
accuracy is {stacking_ac:.5f}
precision is {stacking_prec:.5f}
recall is {stacking_rec:.5f}
"""
)


accuracy is 0.89630
precision is 0.89740
recall is 0.89630



In [55]:
stacking_results = []
best_stacking_model = None
best_stacking_score = (-1, -1)

for n in [2, 3, 4, 5, 6, 7]:
    for passthrough in [False, True]:
        cv = StratifiedKFold(
            n_splits=n,
            shuffle=True,
            random_state=21
        )

        model = StackingClassifier(
            estimators=[
                ("svm", svm_model),
                ("tree", tree_model),
                ("forest", forest_model)
            ],
            final_estimator=OneVsRestClassifier(
                LogisticRegression(
                    solver="liblinear",
                    random_state=21
                )
            ),
            cv=cv,
            passthrough=passthrough
        )

        model.fit(X_train_s, y_train_s)
        pred = model.predict(X_valid_s)

        ac = accuracy_score(y_valid_s, pred)
        prec = precision_score(y_valid_s, pred, average="weighted")
        rec = recall_score(y_valid_s, pred, average="weighted")

        stacking_results.append({
            "n_splits": n,
            "passthrough": passthrough,
            "accuracy": ac,
            "precision": prec,
            "recall": rec
        })

        if (ac, prec) > best_stacking_score:
            best_stacking_score = (ac, prec)
            best_stacking_model = model

stacking_results = pd.DataFrame(stacking_results)
stacking_results = stacking_results.sort_values(
    ["accuracy", "precision"], ascending=False
)

print(
    stacking_results.to_string(
        index=False,
        float_format="{:.5f}".format
    )
)

 n_splits  passthrough  accuracy  precision  recall
        3         True   0.91481    0.91692 0.91481
        6         True   0.91481    0.91661 0.91481
        7         True   0.91111    0.91288 0.91111
        2         True   0.90741    0.90864 0.90741
        4         True   0.90370    0.90683 0.90370
        5         True   0.90370    0.90583 0.90370
        2        False   0.90370    0.90510 0.90370
        3        False   0.89630    0.89740 0.89630
        6        False   0.89630    0.89666 0.89630
        4        False   0.89259    0.89461 0.89259
        5        False   0.89259    0.89365 0.89259
        7        False   0.88889    0.88988 0.88889


In [56]:
stacking_test_pred = best_stacking_model.predict(X_test)

stacking_test_ac = accuracy_score(y_test, stacking_test_pred)
stacking_test_prec = precision_score(
    y_test, stacking_test_pred, average="weighted"
)
stacking_test_rec = recall_score(
    y_test, stacking_test_pred, average="weighted"
)

print(f"n_splits is {best_stacking_model.cv.n_splits}")
print(f"passthrough is {best_stacking_model.passthrough}")

print(
    f"""
test accuracy is {stacking_test_ac:.5f}
test precision is {stacking_test_prec:.5f}
test recall is {stacking_test_rec:.5f}
"""
)

n_splits is 3
passthrough is True

test accuracy is 0.91124
test precision is 0.91306
test recall is 0.91124



Лучший результат получен при n_splits=3 и passthrough=True: accuracy на валидации — 0.91481, precision — 0.91692. При n_splits=6 accuracy такая же, но precision ниже — 0.91661, поэтому выбран вариант с тремя разбиениями.

На тестовой выборке выбранная модель показала accuracy 0.91124, precision 0.91306 и recall 0.91124. Передача исходных признаков итоговой модели (passthrough=True) улучшила качество по сравнению с вариантами, использующими только прогнозы базовых моделей.

## 6. Predictions

1. Choose the best model in terms of accuracy (if there are several of them with equal values, choose the one with the higher precision).
2. Analyze: for which weekday your model makes the most errors (in % of the total number of samples of that class in your full dataset), for which labname and for which users.
3. Save the model.

In [57]:
best_model = best_stacking_model
best_pred = best_model.predict(X_test)

result_df = pd.DataFrame({
    "true_day": y_test,
    "predict_day": best_pred
})
result_df["error"] = result_df["true_day"] != result_df["predict_day"]
errors_by_day = result_df.groupby("true_day")["error"].sum()
total_by_day = df["dayofweek"].value_counts()

error_percent = (errors_by_day / total_by_day * 100).sort_values(ascending=False)
print(error_percent)


true_day
0    6.617647
5    2.214022
4    1.923077
6    1.685393
2    1.342282
3    0.757576
1    0.729927
dtype: float64


In [58]:
# для пользователей
user_columns = df.filter(like="uid_").columns
error_indexes = result_df.index[result_df["error"]]
errors_by_user = df.loc[error_indexes, user_columns].sum()
total_by_user = df[user_columns].sum()
user_error_percent = (
    errors_by_user / total_by_user * 100
).sort_values(ascending=False)
print(user_error_percent)

uid_user_6     16.666667
uid_user_16     6.250000
uid_user_27     4.347826
uid_user_3      4.225352
uid_user_14     3.787879
uid_user_19     3.296703
uid_user_17     2.941176
uid_user_18     2.857143
uid_user_31     2.666667
uid_user_30     2.564103
uid_user_24     1.785714
uid_user_13     1.666667
uid_user_2      1.652893
uid_user_29     1.562500
uid_user_10     1.408451
uid_user_4      1.063830
uid_user_25     0.833333
uid_user_0      0.000000
uid_user_7      0.000000
uid_user_28     0.000000
uid_user_22     0.000000
uid_user_26     0.000000
uid_user_23     0.000000
uid_user_1      0.000000
uid_user_21     0.000000
uid_user_20     0.000000
uid_user_15     0.000000
uid_user_12     0.000000
uid_user_11     0.000000
uid_user_8      0.000000
dtype: float64


In [59]:
lab_df = df.copy().filter(like="labname_")
lab_df["lab"] = lab_df.idxmax(axis=1)
error_sum = lab_df.loc[error_indexes].groupby("lab").sum().max()
sum_lab = lab_df.groupby("lab").sum().max()
lab_error_percent = error_sum / sum_lab * 100
print(lab_error_percent)


labname_code_rvw      3.658537
labname_lab02         0.000000
labname_lab03       100.000000
labname_lab03s      100.000000
labname_lab05s        2.777778
labname_laba04        5.056180
labname_laba04s       3.846154
labname_laba05        0.450450
labname_laba06        2.083333
labname_laba06s       1.639344
labname_project1      0.841220
dtype: float64


In [60]:
joblib.dump(best_model, "../data/ensembles_stacking.joblib")
print("Модель сохранена в ../data/ensembles_stacking.joblib")


Модель сохранена в ../data/ensembles_stacking.joblib


Выбрана модель Stacking с n_splits=3 и passthrough=True: accuracy на валидации — 0.91481, на тесте — 0.91124.

Наибольшая доля ошибок по дням недели: понедельник — 6.62%. По пользователям: user_6 — 16.67%. По лабораторным: lab03, lab03s — 100.00%. Проценты рассчитаны относительно количества объектов каждой группы во всём исходном датасете.

Модель сохранена в data/ensembles_stacking.joblib.